In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from datetime import datetime
import time

service = Service('/usr/bin/chromedriver')
options = Options()
driver = webdriver.Chrome(service=service, options=options)

all_hrefs = []
path = "job_links.txt"

try:
    with open(path, "r") as file:
        existing_links = file.read().splitlines()
except FileNotFoundError:
    existing_links = []

existing_links_set = set(existing_links)

page = 1
while True:
    print(f"Parsing the page {page}...")
    driver.get(f"https://staff.am/jobs?page={page}")

    body = driver.find_element(By.TAG_NAME, 'body')
    for _ in range(4):
        body.send_keys(Keys.PAGE_DOWN)
        time.sleep(0.1)

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "css-175oi2r"))
        )
    except:
        print(f"Element not found on page {page}")
        break

    page_source = driver.page_source
    soup = BeautifulSoup(page_source, "html.parser")
    links = soup.find_all("a")

    hrefs = [link.get("href") for link in links if link.get("href") and link.get("href").startswith("/en/jobs")]
    hrefs = ["https://staff.am" + href for href in hrefs if href]

    # Check for empty page
    if not hrefs:
        print(f"There are no links on page {page}. Parsing stopped.")
        break

    for href in hrefs:
        if href in existing_links_set:
            break
        all_hrefs.append(href)

    else:
        page += 1
        continue

    break


driver.quit()

# Remove duplicates from new links
seen = set()
ordered_hrefs = [x for x in all_hrefs if not (x in seen or seen.add(x))]

# Add new links to the beginning of the file
new_links = [link for link in ordered_hrefs if link not in existing_links_set]
all_links = new_links + existing_links  # New links ahead, old ones moving on

# Save links to a file
with open(path, "w") as file:
    file.write("\n".join(all_links) + "\n")

current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Код успешно завершен в {current_time}.")

print(f"Новые ссылки сохранены в начало файла job_links.txt.")
print(f"Количество новых ссылок: {len(new_links)}")


In [ ]:
import json
import time
from datetime import datetime
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager


SESSION_NUMBER = 10
LINKS_FILE = f"job_links_{SESSION_NUMBER}.txt"
RESULTS_FILE = f"parsing_results_{SESSION_NUMBER}.json"


def parse_job(url, driver):
    driver.get(url)
    time.sleep(3)

    body = driver.find_element(By.TAG_NAME, 'body')
    for _ in range(5):
        body.send_keys(Keys.PAGE_DOWN)
        time.sleep(1)

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "css-175oi2r"))
        )
    except:
        print(f"[!] Элемент не найден: {url}")
        return None

    page_source = driver.page_source
    soup = BeautifulSoup(page_source, "html.parser")
    main_div = soup.find("div", class_="css-175oi2r r-150rngu r-eqz5dr r-16y2uox r-1wbh5a2 r-11yh6sk r-1rnoaur r-agouwx")
    if not main_div:
        print(f"[!] Основной блок не найден: {url}")
        return None

    try:
        next_div = main_div.find("div", class_="css-175oi2r").find("div", class_="css-175oi2r")
        component = next_div.find_all("div", recursive=False)[-1].find_all("div", recursive=False)[-1]
    except:
        print(f"[!] Ошибка разбора структуры страницы: {url}")
        return None

    def safe_text(el):
        return el.get_text(strip=True) if el else None

    try:
        name = safe_text(component.find("h1"))
        qualifications = extract_list(component, 2)
        level = get_simple_text(component, 3)
        prof_skills = extract_tags(component, 4)
        personal_skills = extract_tags(component, 5)
        views = get_views(next_div)
        company = get_company_name(next_div)
        industry = get_industry(next_div)
        extra = get_first_info(component)
    except:
        print(f"[!] Error retrieving data: {url}")
        return None

    job_data = {
        "url": url,
        "name": name,
        "qualifications": qualifications,
        "level": level,
        "professional_skills": prof_skills,
        "personal_skills": personal_skills,
        "views": views,
        "company": company,
        "industry": industry,
        "parsed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    job_data.update(extra)
    return job_data


def extract_list(component, index):
    try:
        comp = component.find_all("div", recursive=False)[1]
        comp = comp.find_all("div", recursive=False)[-1]
        block = comp.find_all("div", class_="css-175oi2r", recursive=False)[index]
        return [el.get_text(strip=True) for el in block.find_all(["p", "li"])]
    except:
        return None

def extract_tags(component, index):
    try:
        comp = component.find_all("div", recursive=False)[1]
        comp = comp.find_all("div", recursive=False)[-1]
        skills = comp.find_all("div", class_="css-175oi2r", recursive=False)[index]
        tags = skills.find_all("div", class_="css-146c3p1")
        return [tag.get_text(strip=True) for tag in tags]
    except:
        return None

def get_simple_text(component, index):
    try:
        comp = component.find_all("div", recursive=False)[1]
        comp = comp.find_all("div", recursive=False)[-1]
        return comp.find_all("div", class_="css-175oi2r", recursive=False)[index].get_text(strip=True)
    except:
        return None

def get_views(next_div):
    try:
        view = next_div.find_all("div", class_="css-175oi2r")[1]
        view = view.find("div", class_="css-175oi2r")
        view = view.find_all("div", class_="css-175oi2r", recursive=False)[-1]
        return view.find("div", class_="css-175oi2r").text.split()[0]
    except:
        return None

def get_company_name(next_div):
    try:
        return next_div.find_all("div", recursive=False)[1].find("div", class_="css-175oi2r").find("div", class_="css-175oi2r").get_text(strip=True)
    except:
        return None

def get_industry(next_div):
    try:
        block = next_div.find_all("div", recursive=False)[1].find("div", class_="css-175oi2r")
        return block.find_all("div", recursive=False)[1].get_text(strip=True)
    except:
        return None

def get_first_info(component):
    try:
        comp = component.find_all("div", recursive=False)[0]
        comp2 = comp.find_all("div", recursive=False)[0]
        comps = comp2.find_all("div", recursive=False)

        geo_and_time = comps[0].find_all("div")
        geo = geo_and_time[2].get_text()
        time_posted = geo_and_time[-1].get_text()
        employment_term = comps[1].find_all("div")[-1].get_text()
        category = comps[2].find_all("div")[-1].get_text()

        return {
            "geo": geo,
            "time": time_posted,
            "Employment term": employment_term,
            "Category": category
        }
    except:
        return {}


def main():
    try:
        with open(LINKS_FILE, "r") as file:
            urls = [line.strip() for line in file if line.strip()]
    except FileNotFoundError:
        print(f"[!] Link file not found: {LINKS_FILE}")
        return

    print(f"🔗 Total links for parsing:  {len(urls)}")

    try:
        with open(RESULTS_FILE, "r", encoding="utf-8") as f:
            results = json.load(f)
    except FileNotFoundError:
        results = []

    options = Options()
    options.headless = True
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)

    for i, url in enumerate(urls, 1):
        print(f"[{i}/{len(urls)}] Parsing: {url}")
        data = parse_job(url, driver)
        if data:
            results.insert(0, data)
        time.sleep(1)

    driver.quit()

    with open(RESULTS_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    print(f"✅ Done! Processed: {len(urls)} job openings.")
    print(f"💾 Results saved in: {RESULTS_FILE}")

# --------------------------------------------------
if __name__ == "__main__":
    main()

#NEW parse links

In [ ]:
import os
import glob
import time
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup


CHROMEDRIVER_PATH = "/usr/bin/chromedriver"
SAVE_DIRECTORY = "/home/mrcomp/Desktop/skills_analysis_project/data/raw_data"
SESSION_NUMBER = 10 


# Collect all old links from job_links_*.txt files
parsed_urls = set()
parsed_files = glob.glob(os.path.join(SAVE_DIRECTORY, "job_links_*.txt"))
for file_path in parsed_files:
    with open(file_path, "r") as f:
        parsed_urls.update(f.read().splitlines())

# Configuring Selenium
service = Service(CHROMEDRIVER_PATH)
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-gpu")
driver = webdriver.Chrome(service=service, options=options)

all_hrefs = []
page = 1

while True:
    print(f"Парсинг страницы {page}...")
    driver.get(f"https://staff.am/jobs?page={page}")

    # Scroll down the page a little
    body = driver.find_element(By.TAG_NAME, 'body')
    for _ in range(4):
        body.send_keys(Keys.PAGE_DOWN)
        time.sleep(0.1)

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "css-175oi2r"))
        )
    except:
        print(f"Element not found on page {page}")
        break

    page_source = driver.page_source
    soup = BeautifulSoup(page_source, "html.parser")
    links = soup.find_all("a")

    hrefs = [link.get("href") for link in links if link.get("href") and link.get("href").startswith("/en/jobs")]
    hrefs = ["https://staff.am" + href for href in hrefs]

    if not hrefs:
        print(f"There are no links on page {page}. Stop.")
        break

    # Checking new links
    stop_parsing = False
    for href in hrefs:
        if href in parsed_urls:
            print(f"The link was already there:{href}")
            stop_parsing = True
            break
        all_hrefs.append(href)

    if stop_parsing:
        break
    else:
        page += 1

driver.quit()

# Delete duplicates
seen = set()
ordered_hrefs = [x for x in all_hrefs if not (x in seen or seen.add(x))]

# Saving new links
if ordered_hrefs:
    filename = os.path.join(SAVE_DIRECTORY, f"job_links_{SESSION_NUMBER}.txt")
    with open(filename, "w") as f:
        f.write("\n".join(ordered_hrefs) + "\n")

    print(f"New {len(ordered_hrefs)} links found.")
    print(f"Saved to file:  {filename}")
else:
    print("No new links found.")

print(f"Code completed in {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
import json
import time
from datetime import datetime
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import undetected_chromedriver as uc
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options


# --------------------------------------------------
#              Path settings
# --------------------------------------------------
SESSION_NUMBER = 10  # 👈 Change the file number manually
LINKS_FILE = f"/home/mrcomp/Desktop/skills_analysis_project/data/raw_data/job_links_{SESSION_NUMBER}.txt"
RESULTS_FILE = f"/home/mrcomp/Desktop/skills_analysis_project/data/raw_data/parsing_results_{SESSION_NUMBER}.json"

# --------------------------------------------------
#              Parsing a single job posting
# --------------------------------------------------

def parse_job(url):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    options.binary_location = "/usr/bin/google-chrome"

    driver = uc.Chrome(options=options)
    driver.get(url)
    time.sleep(3)

    # Page scrolling
    body = driver.find_element(By.TAG_NAME, 'body')
    for _ in range(5):
        body.send_keys(Keys.PAGE_DOWN)
        time.sleep(1)

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "css-175oi2r"))
        )
    except:
        print(f"[!] Element not found: {url}")
        driver.quit()
        return None

    page_source = driver.page_source
    driver.quit()

    soup = BeautifulSoup(page_source, "html.parser")
    main_div = soup.find("div", class_="css-175oi2r r-150rngu r-eqz5dr r-16y2uox r-1wbh5a2 r-11yh6sk r-1rnoaur r-agouwx")
    if not main_div:
        print(f"[!] Main block not found: {url}")
        return None

    try:
        next = main_div.find("div", class_="css-175oi2r").find("div", class_="css-175oi2r")
        component = next.find_all("div", recursive=False)[-1].find_all("div", recursive=False)[-1]
    except Exception as e:
        print(f"[!] Page structure parsing error: {url}")
        return None

    def safe_text(el):
        return el.get_text(strip=True) if el else None

    def get_nested_text(path):
        try:
            for tag in path:
                tag = tag.find("div", class_="css-175oi2r")
            return safe_text(tag)
        except:
            return None
        
    # Extracting information
    try:
        name = safe_text(component.find("h1"))
        qualifications = extract_list(component, 2)
        level = get_simple_text(component, 3)
        prof_skills = extract_tags(component, 4)
        personal_skills = extract_tags(component, 5)
        views = get_views(next)
        company = get_company_name(next)
        industry = get_industry(next)
        extra = get_first_info(component)
    except Exception as e:
        print(f"[!]  Error retrieving data:  {url}")
        return None

    job_data = {
        "url": url,
        "name": name,
        "qualifications": qualifications,
        "level": level,
        "professional_skills": prof_skills,
        "personal_skills": personal_skills,
        "views": views,
        "company": company,
        "industry": industry,
        "parsed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    job_data.update(extra)
    return job_data


# --------------------------------------------------
#              Support functions
# --------------------------------------------------

def extract_list(component, index):
    try:
        comp = component.find_all("div", recursive=False)[1]
        comp = comp.find_all("div", recursive=False)[-1]
        block = comp.find_all("div", class_="css-175oi2r", recursive=False)[index]
        return [el.get_text(strip=True) for el in block.find_all(["p", "li"])]
    except:
        return None

def extract_tags(component, index):
    try:
        comp = component.find_all("div", recursive=False)[1]
        comp = comp.find_all("div", recursive=False)[-1]
        skills = comp.find_all("div", class_="css-175oi2r", recursive=False)[index]
        tags = skills.find_all("div", class_="css-146c3p1")
        return [tag.get_text(strip=True) for tag in tags]
    except:
        return None

def get_simple_text(component, index):
    try:
        comp = component.find_all("div", recursive=False)[1]
        comp = comp.find_all("div", recursive=False)[-1]
        return comp.find_all("div", class_="css-175oi2r", recursive=False)[index].get_text(strip=True)
    except:
        return None

def get_views(next):
    try:
        view = next.find_all("div", class_="css-175oi2r")[1]
        view = view.find("div", class_="css-175oi2r")
        view = view.find_all("div", class_="css-175oi2r", recursive=False)[-1]
        return view.find("div", class_="css-175oi2r").text.split()[0]
    except:
        return None

def get_company_name(next):
    try:
        return next.find_all("div", recursive=False)[1].find("div", class_="css-175oi2r").find("div", class_="css-175oi2r").get_text(strip=True)
    except:
        return None

def get_industry(next):
    try:
        block = next.find_all("div", recursive=False)[1].find("div", class_="css-175oi2r")
        return block.find_all("div", recursive=False)[1].get_text(strip=True)
    except:
        return None

def get_first_info(component):
    try:
        comp = component.find_all("div", recursive=False)[0]
        comp2 = comp.find_all("div", recursive=False)[0]
        comps = comp2.find_all("div", recursive=False)

        geo_and_time = comps[0].find_all("div")
        geo = geo_and_time[2].get_text()
        time_posted = geo_and_time[-1].get_text()
        employment_term = comps[1].find_all("div")[-1].get_text()
        category = comps[2].find_all("div")[-1].get_text()

        return {
            "geo": geo,
            "time": time_posted,
            "Employment term": employment_term,
            "Category": category
        }
    except:
        return {}

# --------------------------------------------------
#                    MAIN
# --------------------------------------------------

def main():
    try:
        with open(LINKS_FILE, "r") as file:
            urls = [line.strip() for line in file if line.strip()]
    except FileNotFoundError:
        print(f"[!] Link file not found:{LINKS_FILE}")
        return

    print(f"🔗 Total links for parsing: {len(urls)}")

    # Loading old data
    try:
        with open(RESULTS_FILE, "r", encoding="utf-8") as f:
            results = json.load(f)
    except FileNotFoundError:
        results = []

    for i, url in enumerate(urls, 1):
        print(f"[{i}/{len(urls)}] Parsing: {url}")
        data = parse_job(url)
        if data:
            results.insert(0, data)
        time.sleep(1)

    with open(RESULTS_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    print(f"✅ Done! Processed: {len(urls)} job openings.")
    print(f"💾 РResults saved in: {RESULTS_FILE}")